# Feedback Analysis

This notebook analyzes the feedback collected from users to understand:
- Score distribution across pipelines
- Training data quality (positive vs negative examples)
- Low-rated questions that need improvement
- Feedback trends over time

In [ ]:
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from collections import Counter
from datetime import datetime

sys.path.append("..")

from src.feedback.collector import get_all_feedback, get_feedback_stats

# Load all feedback
feedback_list = get_all_feedback()
print(f"Total feedback entries: {len(feedback_list)}")

if not feedback_list:
    print("No feedback found. Submit some scores via the chat UI first.")

In [ ]:
# Convert to DataFrame
records = []
for f in feedback_list:
    chunks = json.loads(f.retrieved_chunks) if f.retrieved_chunks else []
    records.append({
        "id": f.id,
        "question": f.question,
        "answer": f.answer,
        "score": f.score,
        "pipeline": f.pipeline,
        "comment": f.comment,
        "chunk_count": len(chunks),
        "created_at": f.created_at,
        "label": "positive" if f.score >= 4 else ("negative" if f.score <= 2 else "neutral"),
    })

df = pd.DataFrame(records)
print(df[["question", "score", "pipeline", "label"]].head(10))

In [ ]:
# Overall stats
stats = get_feedback_stats()
print("=== Feedback Stats ===")
print(f"Total entries:  {stats['total']}")
print(f"Average score:  {stats['avg_score']}")
print(f"Last feedback:  {stats.get('last_feedback_at', 'N/A')}")

label_counts = df["label"].value_counts()
print(f"\nPositive (4-5): {label_counts.get('positive', 0)}")
print(f"Neutral  (3):   {label_counts.get('neutral', 0)}")
print(f"Negative (1-2): {label_counts.get('negative', 0)}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Feedback Analysis Dashboard", fontsize=14, fontweight="bold")

# 1. Score distribution
score_counts = df["score"].value_counts().sort_index()
colors = ["#ff4d6a" if s <= 2 else "#6b7a99" if s == 3 else "#00d2ff" for s in score_counts.index]
axes[0, 0].bar(score_counts.index.astype(int), score_counts.values, color=colors, alpha=0.85, edgecolor="none")
axes[0, 0].set_title("Score Distribution", fontweight="bold")
axes[0, 0].set_xlabel("Score")
axes[0, 0].set_ylabel("Count")
axes[0, 0].set_xticks([1, 2, 3, 4, 5])
axes[0, 0].grid(axis="y", alpha=0.3)

# 2. Training label distribution (pie)
label_colors = {"positive": "#00d2ff", "neutral": "#6b7a99", "negative": "#ff4d6a"}
labels = [k for k in ["positive", "neutral", "negative"] if k in label_counts]
sizes = [label_counts[k] for k in labels]
pie_colors = [label_colors[k] for k in labels]
axes[0, 1].pie(sizes, labels=labels, colors=pie_colors, autopct="%1.1f%%",
               startangle=90, textprops={"fontsize": 11})
axes[0, 1].set_title("Training Data Labels", fontweight="bold")

# 3. Score by pipeline
if df["pipeline"].nunique() > 1:
    pipeline_scores = df.groupby("pipeline")["score"].mean()
    axes[1, 0].bar(pipeline_scores.index, pipeline_scores.values,
                   color=["#00d2ff", "#00e5a0"], alpha=0.85)
    axes[1, 0].set_title("Average Score by Pipeline", fontweight="bold")
    axes[1, 0].set_ylabel("Avg Score")
    axes[1, 0].set_ylim(0, 5)
    axes[1, 0].grid(axis="y", alpha=0.3)
    for i, (pipeline, score) in enumerate(pipeline_scores.items()):
        axes[1, 0].text(i, score + 0.05, f"{score:.2f}", ha="center", fontsize=11)
else:
    axes[1, 0].text(0.5, 0.5, "Only one pipeline used so far",
                    ha="center", va="center", transform=axes[1, 0].transAxes,
                    fontsize=11, color="gray")
    axes[1, 0].set_title("Average Score by Pipeline", fontweight="bold")

# 4. Feedback over time
if "created_at" in df.columns and df["created_at"].notna().any():
    df["date"] = pd.to_datetime(df["created_at"]).dt.date
    daily = df.groupby("date")["score"].agg(["count", "mean"]).reset_index()
    ax2 = axes[1, 1].twinx()
    axes[1, 1].bar(daily["date"], daily["count"], color="#00d2ff", alpha=0.4, label="Count")
    ax2.plot(daily["date"], daily["mean"], color="#ff4d6a", marker="o", linewidth=2, label="Avg Score")
    axes[1, 1].set_title("Feedback Over Time", fontweight="bold")
    axes[1, 1].set_ylabel("Feedback Count", color="#00d2ff")
    ax2.set_ylabel("Avg Score", color="#ff4d6a")
    ax2.set_ylim(0, 5)
    axes[1, 1].xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))
    plt.setp(axes[1, 1].xaxis.get_majorticklabels(), rotation=30, ha="right")
else:
    axes[1, 1].text(0.5, 0.5, "No timestamp data available",
                    ha="center", va="center", transform=axes[1, 1].transAxes,
                    fontsize=11, color="gray")
    axes[1, 1].set_title("Feedback Over Time", fontweight="bold")

plt.tight_layout()
plt.savefig("feedback_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Low-rated questions — most useful for debugging
low_rated = df[df["score"] <= 2].sort_values("score")
print(f"=== Low-rated questions (score <= 2): {len(low_rated)} ===")
for _, row in low_rated.iterrows():
    print(f"\nScore: {int(row['score'])} | Pipeline: {row['pipeline']}")
    print(f"Q: {row['question']}")
    print(f"A: {row['answer'][:200]}...")
    print(f"Comment: {row['comment'] or '(none)'}")

In [ ]:
# Training data quality summary
positive = df[df["score"] >= 4]
negative = df[df["score"] <= 2]

total_positive_samples = positive["chunk_count"].sum()
total_negative_samples = negative["chunk_count"].sum()
total_training_samples = total_positive_samples + total_negative_samples

print("=== Training Data Quality ===")
print(f"Positive examples (label=1.0): {int(total_positive_samples)} chunks from {len(positive)} feedbacks")
print(f"Negative examples (label=0.0): {int(total_negative_samples)} chunks from {len(negative)} feedbacks")
print(f"Neutral (ignored):             {len(df[df['score'] == 3])} feedbacks")
print(f"Total training samples:        {int(total_training_samples)}")

if total_training_samples > 0:
    balance = total_positive_samples / total_training_samples
    print(f"\nClass balance: {balance:.1%} positive / {1-balance:.1%} negative")
    if balance > 0.8:
        print("⚠ Dataset is heavily positive — encourage users to submit low scores too.")
    elif balance < 0.2:
        print("⚠ Dataset is heavily negative — the system may need significant improvement.")
    else:
        print("✓ Dataset balance looks healthy.")

In [ ]:
# Most common question topics (simple word frequency)
from collections import Counter
import re

stopwords = {"what", "how", "is", "a", "the", "an", "and", "or", "of", "in", "does", "are", "do", "can", "it"}
words = []
for q in df["question"]:
    tokens = re.findall(r"\b[a-z]{3,}\b", q.lower())
    words.extend([t for t in tokens if t not in stopwords])

top_words = Counter(words).most_common(15)
print("=== Most Common Question Topics ===")
for word, count in top_words:
    bar = "█" * count
    print(f"  {word:<20} {bar} ({count})")